# Observe the Infrastructure

The agent is running end-to-end. This notebook walks through the observability surfaces the
infrastructure exposes: runtime logs, GenAI traces, gateway audit logs, and AgentCore Memory.

You will need to have run at least one prompt through the agent (notebook 05) before the queries
below return data.

> **Demo logging:** the cells below print raw log events and memory records to make the
> workshop observable. Production systems should log metadata only — never full model
> responses or conversation content.


## Step 1: Open CloudWatch Logs

AgentCore Runtime streams logs to CloudWatch via OpenTelemetry. Open the log group named
`/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent-*-DEFAULT` in the console, or query it
from Python below.

Key log lines to look for:

| Log Entry | What It Confirms |
|-----------|-----------------|
| `[MODEL] Using LiteLLM Gateway: https://...` | Model calls routed through Module 2 |
| `[GATEWAY] URL: https://...` | Tools served by Module 3's gateway |
| `Getting OAuth2 token...` | M2M auth via Token Vault |
| `Negotiated protocol version: 2025-03-26` | MCP handshake with the gateway succeeded |
| `Created session: ...` | AgentCore Memory session created |
| `Started code interpreter in ...` | Code Interpreter sandbox activated |


In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
logs = boto3.client("logs", region_name=REGION)

RUNTIME_LOG_GROUP_PREFIX = "/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent"

matches = logs.describe_log_groups(logGroupNamePrefix=RUNTIME_LOG_GROUP_PREFIX).get("logGroups", [])
if not matches:
    print(f"No log groups matching {RUNTIME_LOG_GROUP_PREFIX}* found — invoke the agent first")
else:
    runtime_group = matches[0]["logGroupName"]
    print(f"Runtime log group: {runtime_group}")

    events = logs.filter_log_events(
        logGroupName=runtime_group,
        limit=10,
    ).get("events", [])
    print(f"Latest {len(events)} events:")
    for e in events[-10:]:
        print(f"  {e['timestamp']}  {e['message'][:180]}")

## Step 2: Inspect the Gateway Audit Log

The Module 3a request interceptor writes a structured audit record for every tool call. Skip
this cell if you are on the Module 3b path (no interceptor on `ac-tools-gateway`).


In [ ]:
INTERCEPTOR_LOG = "/aws/lambda/agentcore-gateway-request-interceptor"

try:
    events = logs.filter_log_events(
        logGroupName=INTERCEPTOR_LOG,
        filterPattern='"tools"',
        limit=5,
    ).get("events", [])
    print(f"Found {len(events)} interceptor audit events")
    for e in events:
        print(f"  {e['timestamp']}  {e['message'][:200]}")
except logs.exceptions.ResourceNotFoundException:
    print(f"{INTERCEPTOR_LOG} not found — you are likely on the AgentCore path (Module 3b). Skip.")


## Step 3: Open the GenAI Observability Dashboard

Amazon Bedrock ships a managed GenAI Observability view in CloudWatch. Open it to see traces
for every agent invocation — each span shows latency, status, and OpenTelemetry metadata.

Open this URL in your browser, then confirm the region selector (top-right of the console) is
set to the region you deployed into:

```text
https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability
```

## Step 4: Query AgentCore Memory

AgentCore Memory persists conversation history per user/session. Query recent events for the
memory resource attached to the runtime.


In [ ]:
cfn = boto3.client("cloudformation", region_name=REGION)
outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName="FAST-stack")["Stacks"][0]["Outputs"]
}

MEMORY_ARN = outputs.get("MemoryArn", "")
if not MEMORY_ARN:
    print("FAST-stack has no MemoryArn output — check the deployment")
else:
    MEMORY_ID = MEMORY_ARN.rsplit("/", 1)[-1]
    print(f"Memory ID: {MEMORY_ID}")

    # Memory events live on the bedrock-agentcore DATA plane (not the
    # bedrock-agentcore-control plane). ListEvents requires memoryId +
    # sessionId + actorId, so we first list actors, then iterate sessions
    # per actor, then fetch a handful of recent events.
    agentcore_data = boto3.client("bedrock-agentcore", region_name=REGION)

    actors = agentcore_data.list_actors(memoryId=MEMORY_ID, maxResults=10).get("actorSummaries", [])
    print(f"Actors with memory: {len(actors)}")
    if not actors:
        print("No memory yet — send a message through the Amplify UI first.")
    else:
        for a in actors[:3]:
            actor_id = a["actorId"]
            sessions = agentcore_data.list_sessions(
                memoryId=MEMORY_ID, actorId=actor_id, maxResults=3,
            ).get("sessionSummaries", [])
            print(f"  actor={actor_id}  sessions={len(sessions)}")
            for s in sessions[:1]:
                session_id = s["sessionId"]
                events = agentcore_data.list_events(
                    memoryId=MEMORY_ID,
                    actorId=actor_id,
                    sessionId=session_id,
                    maxResults=5,
                ).get("events", [])
                print(f"    session={session_id[:12]}  events={len(events)}")
                for ev in events[:3]:
                    print(f"      {ev.get('eventTimestamp', '?')}  {ev.get('eventId', '?')[:12]}")


## What You Observed

| Surface | Shows | Who Uses It |
|---------|-------|-------------|
| Runtime logs | Model invocations, tool calls, errors | AI/ML engineers debugging agent behavior |
| GenAI Observability | End-to-end traces with latency and spans | Platform teams monitoring production |
| Gateway audit log | Every tool invocation with caller identity | Platform teams tracking tool usage |
| AgentCore Memory | Conversation history per user/session | AI/ML engineers tuning memory strategies |

Proceed to **notebook 07 — Register the Agent (optional)** or skip straight to **notebook 08
— Cleanup**.
